In [43]:
import pipit
import pandas as pd

trace = pipit.Trace.from_nsight_sqlite("./nsys-report-a8df.sqlite")
trace._match_events()
display(trace.events)
trace.calc_inc_metrics()


,gpuId,Name,streamId,type,bytes,id,Process,meta,Trace Type,Event Type,Timestamp (ns),_matching_event,_matching_timestamp,_depth,_parent,_children,_kernel_launch
324,<NA>,cuProfilerStart,<NA>,<NA>,<NA>,19829,634992,None,cuda_api,Enter,8337289,2920,8341417,0.0,NaN,None,NaN
2920,<NA>,cuProfilerStart,<NA>,<NA>,<NA>,19829,634992,None,cuda_api,Leave,8341417,324,8337289,NaN,NaN,None,NaN
325,<NA>,cudaEventCreateWithFlags_v3020,<NA>,<NA>,<NA>,19833,634992,None,cuda_api,Enter,8679070,2921,8687657,0.0,NaN,None,NaN
2921,<NA>,cudaEventCreateWithFlags_v3020,<NA>,<NA>,<NA>,19833,634992,None,cuda_api,Leave,8687657,325,8679070,NaN,NaN,None,NaN
326,<NA>,cudaEventRecord_v3020,<NA>,<NA>,<NA>,19837,634992,None,cuda_api,Enter,8693077,2922,8715531,0.0,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3966,<NA>,cudaEventDestroy_v3020,<NA>,<NA>,<NA>,27886,634992,None,cuda_api,Leave,100832280,1370,100831859,NaN,NaN,None,NaN
1371,<NA>,cudaEventDestroy_v3020,<NA>,<NA>,<NA>,27890,634992,None,cuda_api,Enter,100833583,3967,100833963,0.0,NaN,None,NaN
3967,<NA>,cudaEventDestroy_v3020,<NA>,<NA>,<NA>,27890,634992,None,cuda_api,Leave,100833963,1371,100833583,NaN,NaN,None,NaN
1372,<NA>,cudaEventDestroy_v3020,<NA>,<NA>,<NA>,27894,634992,None,cuda_api,Enter,100835095,3968,100835526,0.0,NaN,None,NaN


In [44]:
import numpy as np
import pandas as pd

def collect_full_trace(df, row_idx, seen=None):
    if seen is None:
        seen = set()

    if pd.isna(row_idx) or row_idx in seen or row_idx not in df.index:
        return seen

    seen.add(row_idx)
    row = df.loc[row_idx]

    # Recurse on _children
    children = row['_children']
    if isinstance(children, (list, tuple, np.ndarray)):
        for child_idx in children:
            if pd.notna(child_idx) and int(child_idx) in df.index:
                seen.update(collect_full_trace(df, int(child_idx), seen))
    elif isinstance(children, (int, np.integer)):
        if int(children) in df.index:
            seen.update(collect_full_trace(df, int(children), seen))

    # Recurse on _kernel_launch
    kernel_idx = row.get('_kernel_launch')
    if pd.notna(kernel_idx) and int(kernel_idx) in df.index:
        seen.update(collect_full_trace(df, int(kernel_idx), seen))

    # Recurse on _matching_event
    match_idx = row.get('_matching_event')
    if pd.notna(match_idx) and int(match_idx) in df.index:
        seen.update(collect_full_trace(df, int(match_idx), seen))

    return seen

def extract_prefill_and_related(df):
    prefill_rows = df[
        (df['Name'] == 'Block.Attention.Attention') &
        (df['Trace Type'] == 'nvtx')
    ]
    # Keep only the first 2 rows
    prefill_rows = prefill_rows.head(2)

    df['_kernel_launch'] = df['_kernel_launch'].apply(
        lambda x: int(x) if pd.notna(x) else x
    )


    root_indices = set(prefill_rows.index)

    all_indices = set()
    for idx in root_indices:
        all_indices.update(collect_full_trace(df, idx))
        all_indices.add(idx)

    return df.loc[sorted(all_indices)]

In [45]:
prefill_regions = extract_prefill_and_related(trace.events)
#display(prefill_regions)

# Remove Leave events
prefill_regions = prefill_regions[prefill_regions['type'] != 'Leave']

# Sum "time.inc" for prefill regions grouped by "type"
prefill_time = prefill_regions.groupby('type')['time.inc'].sum()

print(prefill_time)



type
annotation    687360
kernel        673886
Name: time.inc, dtype: Int64


In [ ]:
breakdown = trace.flat_profile(
    parallelism_level=["Process", "gpuId"],
    mapper={"matmul": "ampere", "flash_attn": "flash"},
    ascending=False,
    idle_time=True
)


,,,time.exc
Name,Process,gpuId,
matmul,622453,0,73003018
Other,622453,0,13028818
Idle Time,622453,0,2430532
flash_attn,622453,0,1847672
